In [ ]:
# Hosted D2L setup: fetch the exact helper module used to build this notebook.
from pathlib import Path
from urllib.request import urlretrieve
from importlib.metadata import PackageNotFoundError, version
import importlib.util, os, subprocess, sys

required = ['numpy', 'pandas', 'matplotlib', 'requests', 'scipy', 'pillow', 'regex', 'jax', 'jaxlib', 'flax', 'optax', 'orbax-checkpoint', 'tensorflow', 'protobuf', 'ml-dtypes']
imports = {'pillow': 'PIL', 'orbax-checkpoint': 'orbax', 'protobuf': 'google.protobuf', 'ml-dtypes': 'ml_dtypes'}
pinned = {'jax': ('0.12.0.1', 'jax==0.12.0.1', 'jax[cuda12]==0.12.0.1', 'exact'), 'jaxlib': ('0.12.0.1', 'jaxlib==0.12.0.1', 'jaxlib==0.12.0.1', 'exact'), 'flax': ('0.12.12', 'flax==0.12.12', 'flax==0.12.12', 'exact'), 'optax': ('0.2.8', 'optax==0.2.8', 'optax==0.2.8', 'exact'), 'orbax-checkpoint': ('0.12.0', 'orbax-checkpoint==0.12.0', 'orbax-checkpoint==0.12.0', 'exact')}
fallbacks = {'tensorflow': 'tensorflow==2.21.0', 'protobuf': 'protobuf==7.34.1', 'ml-dtypes': 'ml-dtypes==0.5.4'}
device = os.environ.get("D2L_HOSTED_DEVICE", "auto").lower()
if device not in ("auto", "cpu", "gpu"):
    raise ValueError(f"Invalid D2L_HOSTED_DEVICE={device!r}")
if device == "auto":
    try:
        gpu = (Path("/dev/nvidia0").exists() or
               subprocess.run(["nvidia-smi", "-L"], capture_output=True,
                              timeout=5).returncode == 0)
    except (FileNotFoundError, subprocess.SubprocessError):
        gpu = False
else:
    gpu = device == "gpu"
if not gpu:
    os.environ.setdefault("CUDA_VISIBLE_DEVICES", "-1")
    os.environ.setdefault("JAX_PLATFORMS", "cpu")
tensorflow_version = None
if 'jax' in ("tensorflow", "jax"):
    try:
        tensorflow_version = version("tensorflow")
    except PackageNotFoundError:
        pass
# Colab's CPU image currently carries a CUDA-enabled TensorFlow wheel. Its
# first ordinary tensor operation probes CUDA and emits an error-level cuInit
# diagnostic. JAX notebooks also use TensorFlow for data loading, so overlay
# the matching CPU build in both CPU variants. Keep the provider's
# ``tensorflow`` distribution metadata: other preinstalled Colab packages
# depend on that distribution name, while both wheels expose the same module.
if not gpu and 'jax' in ("tensorflow", "jax"):
    try:
        tensorflow_cpu_version = version("tensorflow-cpu")
    except PackageNotFoundError:
        tensorflow_cpu_version = None
    if (tensorflow_version is not None and
            tensorflow_cpu_version != tensorflow_version):
        subprocess.check_call([
            sys.executable, "-m", "pip", "install", "-q", "--no-deps",
            f"tensorflow-cpu=={tensorflow_version}",
        ])
if "tf-keras" in fallbacks and tensorflow_version is not None:
    fallbacks["tf-keras"] = f"tf-keras=={tensorflow_version}"
missing = []
for package in required:
    if package in pinned:
        wanted, cpu_requirement, gpu_requirement, match = pinned[package]
        requirement = gpu_requirement if gpu else cpu_requirement
        try:
            installed = version(package)
        except PackageNotFoundError:
            installed = None
        actual = (installed.split("+", 1)[0]
                  if installed is not None and match == "public" else installed)
        if actual != wanted:
            missing.append(requirement)
    elif importlib.util.find_spec(imports.get(package, package)) is None:
        missing.append(fallbacks.get(package, package))
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

mismatched = []
for package, (wanted, _, _, match) in pinned.items():
    try:
        installed = version(package)
    except PackageNotFoundError:
        installed = None
    actual = (installed.split("+", 1)[0]
              if installed is not None and match == "public" else installed)
    if actual != wanted:
        mismatched.append(f"{package}={installed!r} (expected {wanted})")
if mismatched:
    raise RuntimeError("Hosted runtime setup failed: " + ", ".join(mismatched))

root = Path(".d2l-hosted") / "8a5255b0fce49fc730baae6ea6ecc601d45bb16b"
package = root / "d2l"
package.mkdir(parents=True, exist_ok=True)
base = "https://raw.githubusercontent.com/smolix/d2l-neu/8a5255b0fce49fc730baae6ea6ecc601d45bb16b/d2l"
for name in ('__init__.py', 'jax.py'):
    target = package / name
    if not target.exists():
        urlretrieve(f"{base}/{name}", target)
if str(root.resolve()) not in sys.path:
    sys.path.insert(0, str(root.resolve()))
pythonpath = os.environ.get("PYTHONPATH", "").split(os.pathsep)
if str(root.resolve()) not in pythonpath:
    os.environ["PYTHONPATH"] = os.pathsep.join(
        [str(root.resolve()), *[entry for entry in pythonpath if entry]]
    )


# Selective State Space Models

The S4D layer of that section treats every input alike: an input from
$k$ steps ago reaches the output through the fixed number $K_k$ of the
convolution kernel, whatever that input was. Whether the token ten steps
ago was the key fact of a paragraph or a comma, its influence on the present
is the same. Language is not like this. Most tokens can be forgotten
quickly, and a few, such as a name, a number, or the subject of a sentence,
must be kept for a long time, at positions that differ from one sentence to
the next. A memory that keeps everything for the same time fills up with
commas; a memory that forgets quickly loses the name. Can a state space
model decide, token by token, what to keep and for how long?

Two models of the previous section each have half of a solution. The minGRU
chooses its decay from the current input, so it can keep one token and
overwrite the next, but its gates start with a memory of about one step,
and its state holds one number per channel. S4D starts with memories of a
step to a thousand steps and holds $N$ numbers per channel, but its dynamics
are the same for every input. *Selective* state space models, introduced
with the Mamba architecture by @Gu.Dao.2023, combine the two: the
step size $\Delta$ of the discretized SSM, and the vectors that write to and
read from the state, become functions of the current input. A small step
size keeps the state and ignores the token; a large one overwrites the state
with it. With input-dependent dynamics, Mamba matched or exceeded
transformers of the same size in language modeling. The convolution of S4D
no longer applies, because the dynamics change from step to step, but the
scan of that section does. We test these models on a task that
requires choosing what to keep, build the Mamba layer, measure where its
state comes from, and train it as a language model. The state of a selective SSM turns out to be a matrix written one outer
product at a time, which the next section studies in its own right.

In [ ]:
import math
import numpy as np
import jax
from jax import numpy as jnp
from flax import nnx
import optax
from d2l import jax as d2l

In [ ]:
import random

## The Selectivity Problem

### Selective Copying

The *selective copying* task of @Gu.Dao.2023 isolates the decision
of what to keep (the figure). A sequence consists of
1,024 filler tokens among which four symbols, drawn from eight, sit at
random positions. Four query tokens follow, and at each of them the model
must output the next symbol, in order. Chance accuracy is 1/8. A solution
must store a token because it is a symbol rather than filler, at whatever
position it occurs, and keep it for up to a thousand steps; weights that
depend only on position cannot do this, since the same position holds
filler in one sequence and a symbol in the next. @Gu.Dao.2023 use a
larger version of the task, with longer sequences and more symbols to copy.

![The selective copying task. A few marked symbols (color) are scattered among filler tokens (grey) at positions that change from sequence to sequence; at the query tokens, the model must output the symbols in order.](https://raw.githubusercontent.com/smolix/d2l-neu/notebooks/img/mdl-modernrnn-selective-copy.svg)

Token 0 is the filler, token 1 the query, and tokens 2 to 9 the symbols. We
draw a fresh batch of sequences for every training step, so the model
cannot memorize the training data.

In [2]:
def copy_batch(key, batch_size, num_fillers=1024, num_marked=4,
               num_symbols=8):
    """Symbols at random positions among fillers, then query tokens."""
    key_y, key_p = jax.random.split(key)
    Y = jax.random.randint(key_y, (batch_size, num_marked), 2, 2 + num_symbols)
    positions = jnp.argsort(jax.random.uniform(key_p, (batch_size, num_fillers)), 1)
    X = jnp.zeros((batch_size, num_fillers + num_marked), dtype=jnp.int32)
    X = X.at[:, num_fillers:].set(1)  # Query tokens
    rows = jnp.arange(batch_size)[:, None]
    X = X.at[rows, jnp.sort(positions[:, :num_marked], 1)].set(Y)
    return X, Y

X, Y = copy_batch(jax.random.key(random.getrandbits(32)), 2, num_fillers=12)
print(X, Y, sep='\n')

[[0 0 5 0 0 0 0 0 0 8 7 9 1 1 1 1]
 [8 0 0 0 0 0 0 3 0 8 0 4 1 1 1 1]]
[[5 8 7 9]
 [8 3 8 4]]


We train every model as `d2l.SeqLM`, the residual stack of
that section, with two blocks of width 48, and read its predictions at
the four query positions. Training uses the loop `d2l.train_queries` of
that section, with Adam at learning rate 0.003 and
gradient clipping, for 2,500 steps of 64 sequences; accuracy is measured on
1,024 new sequences. We start with S4D and the minGRU.

In [ ]:
def train_copy(model, num_steps=2500, lr=3e-3):
    """Train on fresh batches of selective copying; return test accuracy."""
    optimizer = nnx.Optimizer(model, optax.chain(
        optax.clip_by_global_norm(1.0), optax.adam(lr)), wrt=nnx.Param)
    key = jax.random.key(random.getrandbits(32))  # New data every run
    d2l.train_queries(model, lambda step: copy_batch(
        jax.random.fold_in(key, step), 64), num_steps, optimizer)
    X, Y = copy_batch(jax.random.fold_in(key, num_steps), 1024)
    return float(jnp.stack([d2l.query_accuracy(model, X[i:i + 128],  # In parts
                                               Y[i:i + 128])
                            for i in range(0, 1024, 128)]).mean())

In [4]:
rngs = nnx.Rngs(random.getrandbits(32))
models = {
    'S4D': d2l.SeqLM(10, 48, [d2l.S4D(48, rngs=rngs) for _ in range(2)],
                     rngs=rngs),
    'minGRU': d2l.SeqLM(10, 48, [d2l.MinGRU(48, 48, rngs=rngs)
                                 for _ in range(2)], rngs=rngs)}
for name, model in models.items():
    print(f'{name:8s} accuracy {train_copy(model):.2f}')

S4D      accuracy 0.20


minGRU   accuracy 0.99


S4D answers about one query in five correctly, a little above the
chance rate of one in eight: its weights depend on distance, and the same
distance holds a symbol in one sequence and a filler in the next. The minGRU
answers nine in ten or more in most runs, and rarely fewer than eight. Its
decay is chosen by each input, so it can keep a symbol and let fillers pass,
and it nearly solves the task. Its
gates, however, start near one half: on the delayed recall task of
that section, which needs no selection at all, the minGRU could not
keep a symbol across 1,000 fillers unless its gates were initialized with
long memories. Selective copying is easier in this respect, because a symbol
sometimes sits close to a query, and the short-range gradients from such
sequences teach the gates to keep symbols longer and longer.

## Selective Dynamics

### Input-Dependent Step Sizes

Recall the zero-order hold of that section: a state that obeys
$h'(t) = \lambda h(t) + x(t)$ with rate $\lambda < 0$, read at steps of
length $\Delta$, decays by $e^{\Delta\lambda}$ per step, and the input enters
with weight $(e^{\Delta\lambda} - 1)/\lambda$, which is close to $\Delta$
when $\Delta|\lambda|$ is small. A small step keeps the state and admits
little of the input; a large step forgets the state and admits much. The
step size acts as a gate. In S4D it was a learned constant per channel; a
selective SSM computes it from the input.

Let the input of the layer have $H$ channels, $\mathbf{x}_t \in
\mathbb{R}^H$, and let each channel $c$ have a state $\mathbf{h}_t^{(c)} \in
\mathbb{R}^N$ with rates $\lambda_1, \ldots, \lambda_N$, initialized to
$-1, \ldots, -N$ as in S4D. The selective SSM computes three quantities from
the input at every step: a step size per channel, and a write vector and a
read vector shared by all channels,

$$\boldsymbol{\Delta}_t = \textrm{softplus}(\mathbf{W}_\Delta \mathbf{x}_t +
\mathbf{b}_\Delta), \qquad \mathbf{B}_t = \mathbf{W}_B \mathbf{x}_t, \qquad
\mathbf{C}_t = \mathbf{W}_C \mathbf{x}_t,$$

where softplus$(z) = \log(1 + e^z)$ keeps the step sizes positive. The state
of each channel then follows the discretized SSM with these step-dependent
values,

$$
\mathbf{h}_t^{(c)} = e^{\Delta_{t,c} \boldsymbol{\lambda}} \odot
\mathbf{h}_{t-1}^{(c)} + \Delta_{t,c}\, x_{t,c}\, \mathbf{B}_t, \qquad
y_{t,c} = \mathbf{C}_t^\top \mathbf{h}_t^{(c)} + D_c\, x_{t,c}.
$$

Here $e^{\Delta_{t,c}\boldsymbol{\lambda}}$ is the vector of decays of the
$N$ states, and the input weight is the approximation $\Delta$ of the
zero-order hold, which Mamba uses. Compared with S4D, three things change,
and each is a decision the layer now makes per token. The step size
$\Delta_{t,c}$ decides how much channel $c$ forgets and how much of the
token it writes; at a filler token the layer can set it near zero and skip
the token, and at a symbol it can set it large and store it. The write
vector $\mathbf{B}_t$ decides where in the $N$-dimensional state the token
goes, and the read vector $\mathbf{C}_t$ decides what the output reads.
Compared with the minGRU, the decay is still chosen by the input, but each
channel has $N$ states with rates that span a range, and the step sizes are
initialized log-uniformly between 0.001 and 0.1, so that the layer starts
with memories from a few steps to a thousand.

### Training with a Scan

The selective SSM cannot be computed as a convolution: the weight of an
input from $k$ steps ago is now a product of $k$ different decays, which
depends on the tokens in between, so there is no single kernel. The update
the equation is still affine in the state, with decays and
writes that the layer computes for all positions before the recurrence
starts, so `d2l.linear_scan` computes all states in parallel, exactly as for
the minGRU.

The scan has a cost that the minGRU did not have: memory. The state at each
position now has $H \times N$ numbers, 1,536 for 96 channels with 16 states
(the language model below has 16 states per channel),
and a minibatch of 64 sequences of 1,024 steps holds 100 million of them,
400 MB in 32-bit floats. Backpropagation through the doubling scan stores
every round, $\lceil \log_2 T \rceil = 10$ of them, which takes several
gigabytes per layer. A property of linear recurrences avoids this. Let
$\mathbf{g}_t$ be the derivative of the loss with respect to the state
$\mathbf{h}_t$. The state $\mathbf{h}_t$ affects the loss directly, through
the output at step $t$, and through the next state, $\mathbf{h}_{t+1} =
\mathbf{a}_{t+1} \odot \mathbf{h}_t + \mathbf{b}_{t+1}$, so

$$\mathbf{g}_t = \frac{\partial \ell}{\partial \mathbf{h}_t}\Big|_\textrm{direct}
+ \mathbf{a}_{t+1} \odot \mathbf{g}_{t+1}.$$

This is again a linear recurrence, run backward in time, and a second scan
computes it. The gradients of the decays and writes follow directly:
$\partial\ell/\partial\mathbf{b}_t = \mathbf{g}_t$ and
$\partial\ell/\partial\mathbf{a}_t = \mathbf{g}_t \odot \mathbf{h}_{t-1}$.
The backward pass then needs only the decays and the states, not the
intermediate rounds. Mamba's fused GPU kernel goes further: it never writes
the states to the GPU's main memory at all, but computes them in fast on-chip
memory, reads out $\mathbf{C}_t^\top \mathbf{h}_t$ there, and recomputes
the states during the backward pass.

In code, the backward pass is a second call of `linear_scan` on the
reversed sequence. We give the scan this gradient explicitly, as a custom
backward function, and check it against the gradient that automatic
differentiation computes through the rounds of the scan.

In [ ]:
@jax.custom_vjp
def _adjoint_scan(a, b):
    return d2l.linear_scan(a, b)

def _adjoint_scan_forward(a, b):
    """Forward pass: the states, and what the backward pass needs."""
    h = d2l.linear_scan(a, b)
    return h, (a, h)

def _adjoint_scan_backward(saved, grad_h):
    """Backward pass: gradients of a and b by a scan in reverse time."""
    a, h = saved
    # g_t = grad_h_t + a_{t+1} g_{t+1}: a linear recurrence from the end
    a_next = jnp.concatenate([a[:, 1:], jnp.ones_like(a[:, :1])], 1)
    g = d2l.linear_scan(a_next[:, ::-1], grad_h[:, ::-1])[:, ::-1]
    h_prev = jnp.concatenate([jnp.zeros_like(h[:, :1]), h[:, :-1]], 1)
    return g * h_prev, g

_adjoint_scan.defvjp(_adjoint_scan_forward, _adjoint_scan_backward)

def adjoint_scan(a, b):
    """All states of h_t = a_t * h_{t-1} + b_t, differentiated by a scan."""
    return _adjoint_scan(jnp.broadcast_to(a, b.shape), b)

In [6]:
key_a, key_b, key_w = jax.random.split(jax.random.key(random.getrandbits(32)),
                                       3)
a = jax.random.uniform(key_a, (2, 100, 8))
b = jax.random.normal(key_b, (2, 100, 8))
w = jax.random.normal(key_w, (2, 100, 8))  # A loss that weights every state
grad = lambda scan: jax.grad(lambda a, b: (scan(a, b) * w).sum(), (0, 1))
deviation = max(float(jnp.abs(g1 - g2).max()) for g1, g2 in
                zip(grad(d2l.linear_scan)(a, b), grad(adjoint_scan)(a, b)))
print(f'max deviation: {deviation:.1e}')

max deviation: 9.5e-07


The two gradients agree to rounding error. The adjoint scan stores two
tensors of the size of the states instead of ten, and in PyTorch it lets the
models below train in a few gigabytes.

## The Mamba Layer

### The Layer

A selective SSM mixes information along the sequence, but each channel
separately and linearly. The Mamba layer [@Gu.Dao.2023] surrounds it
with projections, a nonlinearity, and a gate, which mix the channels as the
MLP of a transformer block does (the figure). An input projection widens the $d$-dimensional
input to $H = 2d$ channels and produces a second branch of the same width. A
short causal convolution over the last four tokens lets each channel combine
a token with its neighbors before the SSM sees it, followed by a SiLU
nonlinearity. The SSM output is multiplied by the SiLU of the second branch,
a gate, and an output projection returns to width $d$.

![The Mamba layer. The input is projected to two branches of width $H = 2d$; one passes through a short causal convolution, a SiLU, and the selective SSM, whose step sizes and write and read vectors are computed from its input; the other gates the result. A final projection returns to width $d$.](https://raw.githubusercontent.com/smolix/d2l-neu/notebooks/img/mdl-modernrnn-mamba-block.svg)

The selective SSM holds the rates of its states, three projections that
compute the step sizes and the write and read vectors from the input, and
the skip weights $D$. As in @Gu.Dao.2023, the step sizes come from a
projection of low rank $R = \lceil H/16 \rceil$, down to $R$ numbers and
back up to the $H$ channels: the step sizes of the channels move together,
and the projection costs $2HR$ weights instead of $H^2$. The step sizes are
initialized as in S4D: the up-projection starts with small weights, uniform
in $\pm R^{-1/2}$, and the biases so that the softplus of the bias alone
gives step sizes log-uniformly distributed between `dt_min` and `dt_max`.
Training then learns how the step sizes depend on the input.

In [7]:
class SelectiveSSM(nnx.Module):
    """A diagonal SSM whose step sizes and write and read vectors depend on
    the input: N states for each of num_channels channels."""
    def __init__(self, num_channels, num_states=16, dt_min=1e-3, dt_max=1e-1,
                 *, rngs):
        H, N, R = num_channels, num_states, math.ceil(num_channels / 16)
        self.log_neg_rate = nnx.Param(  # Rates -1, ..., -N in every channel
            jnp.tile(jnp.log(jnp.arange(1., N + 1)), (H, 1)))
        def dt_bias(key, shape, dtype=jnp.float32):  # softplus(bias) = dt
            dt = dt_min * (dt_max / dt_min) ** jax.random.uniform(key, shape)
            return (dt + jnp.log(-jnp.expm1(-dt))).astype(dtype)
        def small(key, shape, dtype=jnp.float32):  # U(-R^-1/2, R^-1/2)
            return jax.random.uniform(key, shape, dtype, -R ** -0.5, R ** -0.5)
        self.W_dt = nnx.Sequential(  # Step size per channel, through rank R
            nnx.Linear(H, R, use_bias=False, rngs=rngs),
            nnx.Linear(R, H, kernel_init=small, bias_init=dt_bias, rngs=rngs))
        self.W_B = nnx.Linear(H, N, use_bias=False, rngs=rngs)  # Write vector
        self.W_C = nnx.Linear(H, N, use_bias=False, rngs=rngs)  # Read vector
        self.D = nnx.Param(jnp.ones(H))

The method `log_coefficients` computes the step sizes from the input and turns
them into the decays $e^{\Delta_{t,c}\lambda_n}$, kept as logarithms, and
the writes $\Delta_{t,c}\, x_{t,c}\, \mathbf{B}_t$ of
the equation for every position, channel, and state. The
forward pass scans them and reads every channel's state with
$\mathbf{C}_t$.

In [ ]:
@d2l.add_to_class(SelectiveSSM)
def log_coefficients(self, U):
    """Log-decays and writes, each of shape (batch_size, T, H, N)."""
    dt = jax.nn.softplus(self.W_dt(U))  # Step sizes, (batch_size, T, H)
    log_a = dt[..., None] * -jnp.exp(self.log_neg_rate[...])
    return log_a, (dt * U)[..., None] * self.W_B(U)[..., None, :]

@d2l.add_to_class(SelectiveSSM)
def __call__(self, U):  # U: (batch_size, num_steps, num_channels)
    log_a, b = self.log_coefficients(U)
    H = adjoint_scan(jnp.exp(log_a), b)  # States, (batch_size, T, H, N)
    return (H * self.W_C(U)[..., None, :]).sum(-1) + self.D[...] * U

The Mamba layer wraps the selective SSM as described above; like the
reference implementation, its two projections have no biases. Its
interface is that of the other mixers of the chapter, so it can be used in
`d2l.SeqLM`. @Gu.Dao.2023 stack Mamba layers directly, without the
MLP between them; we keep the MLP of `d2l.SeqLM`, so that every mixer of
the chapter sits in the same stack.

In [ ]:
class Mamba(nnx.Module):
    """The Mamba layer: a gated selective SSM between two projections."""
    def __init__(self, num_hiddens, num_states=16, expand=2, conv_width=4,
                 *, rngs, **ssm_args):
        H = expand * num_hiddens
        self.W_in = nnx.Linear(num_hiddens, 2 * H, use_bias=False, rngs=rngs)
        self.conv = nnx.Conv(H, H, (conv_width,), feature_group_count=H,
                             padding=((conv_width - 1, 0),), rngs=rngs)  # Causal
        self.ssm = SelectiveSSM(H, num_states, rngs=rngs, **ssm_args)
        self.W_out = nnx.Linear(H, num_hiddens, use_bias=False, rngs=rngs)

    def ssm_input(self, X):
        """The input of the SSM and the gate branch."""
        U, Z = jnp.split(self.W_in(X), 2, -1)
        return jax.nn.silu(self.conv(U)), Z

    def __call__(self, X):  # X: (batch_size, num_steps, num_hiddens)
        U, Z = self.ssm_input(X)
        return self.W_out(self.ssm(U) * jax.nn.silu(Z))

### Selective Copying with Mamba

We train a Mamba model with four states per channel, with the setup above.

In [10]:
rngs = nnx.Rngs(random.getrandbits(32))
models['Mamba'] = d2l.SeqLM(10, 48, [Mamba(48, num_states=4, rngs=rngs)
                                     for _ in range(2)], rngs=rngs)
print(f"Mamba    accuracy {train_copy(models['Mamba']):.2f}")

Mamba    accuracy 1.00


Mamba answers nearly every query correctly. It combines the two properties that
the models of that section had separately: decays chosen by the input,
as in the minGRU, and memories of many lengths from the start, as in S4D.
The minGRU, which has only the first, nearly solves the task; S4D, which
has only the second, does not come close.

### Contributions to the State

Accuracy shows that Mamba keeps the symbols; it does not show how. Because
the state is a sum of past writes, each decayed by the decays met since, we
can measure how much each input contributes to it. Unrolling the recurrence
as in that section, the state at step $T$ is $\sum_{s \le T}
(\prod_{r = s+1}^{T} \mathbf{a}_r) \odot \mathbf{b}_s$, and the size of
the term for position $s$ is its contribution. With the logarithms of the
decays, each product is the exponential of a difference of cumulative sums.

In [11]:
def state_contributions(log_a, b):
    """Size of each position's term in the final state of h = a h + b."""
    cum = log_a.cumsum(1)
    terms = jnp.exp(cum[:, -1:] - cum) * b  # Decayed to the last step
    return jnp.linalg.norm(terms.reshape(*terms.shape[:2], -1), axis=-1)

We apply it to the first layer of the trained S4D and Mamba models, on one
test sequence, for the state at the first query token. For Mamba,
`log_coefficients` returns the log-decays and writes at every position. We
give S4D the same method, with the decays of the equation, the same
at every step, so that the measurement treats the two layers alike.

In [ ]:
@d2l.add_to_class(d2l.S4D)
def log_coefficients(self, X):
    """Log-decays and writes of the recurrence, each (batch_size, T, d, N)."""
    rate = -jnp.exp(self.log_neg_rate[...])
    log_a = jnp.exp(self.log_dt[...]) * rate  # log a_bar = dt * lambda
    b_bar = jnp.expm1(log_a) / rate
    return (jnp.broadcast_to(log_a, (*X.shape, log_a.shape[-1])),
            b_bar * X[..., None])

In [ ]:
X, _ = copy_batch(jax.random.key(random.getrandbits(32)), 1)
X = X[:, :-3]  # Up to the first query token
s4d, mamba = models['S4D'], models['Mamba']  # The first layer of each model
c_s4d = state_contributions(
    *s4d.mixers[0].log_coefficients(s4d.first_mixer_input(X)))[0]
U, _ = mamba.mixers[0].ssm_input(mamba.first_mixer_input(X))
c_mamba = state_contributions(*mamba.mixers[0].ssm.log_coefficients(U))[0]

We also compute the share of the state that comes from the four positions
starting at each symbol, which cover less than 2% of the sequence: the
short convolution in front of the SSM spreads a token over the next three
positions, so the layer can write it at any of them.

In [ ]:
def symbol_windows(X, width=4):
    """Positions of each symbol and the width - 1 after it, without repeats."""
    starts = jnp.nonzero(X[0] > 1)[0]  # The four symbols
    windows = (starts[:, None] + jnp.arange(width)).flatten()
    windows = jnp.unique(windows)  # Windows of nearby symbols may overlap
    return windows[windows < X.shape[1]]  # A window may reach past the end

windows = symbol_windows(X)
for name, c in [('S4D', c_s4d), ('Mamba', c_mamba)]:
    print(f'{name:6s} share from the symbols: {c[windows].sum() / c.sum():.2f}')
d2l.plot(jnp.arange(X.shape[1]), [c / c.max() for c in (c_s4d, c_mamba)],
         'position', 'contribution to the state', legend=['S4D', 'Mamba'],
         yscale='log', figsize=(6, 3))

For S4D, the contributions rise smoothly toward the query, and the most
recent inputs dominate, whatever they are. A symbol may stand out from the
fillers next to it, because the size of a write depends on the token, but
its decay does not: a symbol hundreds of steps back still contributes far
less than the last few fillers. Unless a symbol happens to sit among the
last few positions, the windows at the symbols supply only a few percent of
the state, about their share of the positions. For Mamba, the
contributions spike at the four symbols, far above the fillers around them,
wherever the symbols sit, and the windows supply many times their share of
the positions: the selective SSM writes the symbols and lets the fillers
pass. The task does not determine how much of the state they supply. Runs
that copy equally well find different solutions, some reserving nearly the
whole state for the symbols, others leaving most of it to the last few
positions before the query, so the printed share changes widely from run
to run.

### Language Modeling

We train the Mamba layer as the mixer of `d2l.SeqLM` on the
character-level Penn Treebank, with the recipe of the chapter: two blocks of
width 128, so that each layer has $H = 256$ channels with 16 states, and
3,000 steps of AdamW through `d2l.fit_lm`. With this recipe the minGRU
reached a perplexity of about 3.2 and S4D about 3.35.

In [13]:
data = d2l.PennTreebank(batch_size=64, num_steps=128)

In [14]:
rngs = nnx.Rngs(random.getrandbits(32))
model = d2l.SeqLM(len(data.vocab), 128,
                  [Mamba(128, rngs=rngs) for _ in range(2)], rngs=rngs)
print(f'Mamba perplexity: {d2l.fit_lm(model, data):.2f}')

Mamba perplexity: 3.01


Mamba reaches a perplexity of about 3.0, level with the LSTM and the GRU
and well below the minGRU and S4D, with fewer parameters than the LSTM
model, about 520,000 against 540,000. The low-rank step sizes make the
difference: a dense projection of $H \times H$ weights would add about
115,000. A model trained entirely with scans, and generating with a state of
fixed size, models text as well as the gated networks that dominated before
transformers.

The state of a selective SSM is an $H \times N$ matrix for every layer, and
each step adds to it the outer product of the step-scaled input with the
write vector $\mathbf{B}_t$; the read vector $\mathbf{C}_t$ multiplies it.
This is a memory of key–value pairs, with $\mathbf{B}_t$ as key and
$\mathbf{C}_t$ as query. The next section studies such memories directly:
how many pairs they hold, and how to train them with matrix products.

## Summary

A state space model with fixed dynamics weights its inputs by their
distance, so it cannot keep a token because of what the token is, and it
fails at selective copying. A selective state space model computes its step
sizes, and the vectors that write to and read from its state, from the
current input. A small step size skips a token and a large one stores it, so
the step size acts as the gate of a gated recurrent network, while the state
keeps $N$ numbers per channel and starts with the multi-timescale
initialization of S4D. Mamba solves selective copying over a thousand filler
tokens, which the minGRU nearly solves and S4D does not. In the trained
model, the few positions after each symbol supply a part of the state at
the query many times larger than their share of the positions. The
selective model can no longer be computed as a convolution, but its update
is still affine in the state, so a scan trains it, and the gradient of the
scan is a second scan, run backward. The Mamba layer wraps
the selective SSM in projections, a short convolution, and a gate, and it is
a strong language model at this scale.

## Exercises

1. **A step size is a gate.** Consider one channel with one state, rate
   $\lambda = -1$, and the exact zero-order hold of that section, so
   that $h_t = e^{-\Delta_t} h_{t-1} + (1 - e^{-\Delta_t}) x_t$.
    1. Show that with $\Delta_t = \textrm{softplus}(s_t)$ the update is
       $h_t = z_t h_{t-1} + (1 - z_t) x_t$ with $z_t = \sigma(-s_t)$, the
       blend of a GRU with update gate $z_t$.
    1. What does the approximation $\Delta_t$ of the input weight, which
       Mamba uses, change? For which $\Delta_t$ is it accurate?

1. **Memory of the scan.** For a Mamba layer with $H$ channels and $N$
   states, a minibatch of $B$ sequences of length $T$, and 32-bit numbers,
   count the bytes that backpropagation stores for the scan, (a) through the
   rounds of the doubling scan, (b) with the adjoint scan, and (c) with a
   kernel that stores only the inputs of the layer and recomputes the
   states. Evaluate the three for $H = 4096$, $N = 16$, $B = 8$, and $T =
   8192$, and compare with the memory of a GPU.

1. [code] **Causality.** The convolution of the Mamba layer must see only
   the current and earlier tokens; the code pads the sequence so that
   output $t$ depends on inputs $t-3, \ldots, t$.
    1. Change a token in the middle of a random input sequence and check
       that the outputs at earlier positions do not change.
    1. Shift the padding so that output $t$ also sees input $t+1$. What
       goes wrong, and how does the model's training loss on the copying
       task react?

1. [code] **Longer sequences.** Evaluate the Mamba model trained on 1,024
   fillers on sequences with 2,048 and 4,096 fillers, without retraining.
    1. Does it still copy the symbols? Compare with the minGRU.
    1. For each channel of the first layer, compute the decay of its slowest
       state over 1,000 filler tokens. How many channels can carry a symbol
       across 4,096 fillers?

1. **Weights that depend on position.** ● Consider any model whose state
   at step $T$ is $\sum_{s \le T} \mathbf{w}_{T-s} \odot f(x_s)$, with
   fixed weights $\mathbf{w}_k$ and a fixed function $f$ of a single
   token, and whose output is any function of this state. S4D is of this
   form. Show that for selective copying with two symbols among fillers,
   the state cannot depend on the order of the symbols unless the weights
   $\mathbf{w}_k$ differ for different distances $k$, and that the model
   must then distinguish, from the state alone, every pair of positions.
   Why does this make the task hard for S4D as the number of fillers
   grows, while a selective model can keep its state unchanged on fillers?

1. [code] **Which part must be selective?** ●
    1. Train three variants of the Mamba model on selective copying: with
       input-dependent step sizes but fixed write and read vectors
       $\mathbf{B}$ and $\mathbf{C}$, with fixed step sizes but
       input-dependent $\mathbf{B}_t$ and $\mathbf{C}_t$, and with
       neither. Which variants solve the task? Relate the result to the
       contributions to the state measured above.
    1. In the trained Mamba model, set the step sizes of the first layer to
       their median over filler positions at the four positions after each
       symbol, so that the layer cannot write the symbols with a large
       step. How much does the accuracy drop? Repeat with the step sizes
       set to their median over these positions everywhere else, so that
       the layer cannot skip the fillers.

1. [code] **Generation.** Add a method `step` to `SelectiveSSM` and to
   `Mamba` that processes one token, given the state of the SSM and the
   last three inputs of the convolution, and returns the output and the
   new state. Check that stepping through a sequence reproduces the
   outputs of the forward pass. How many numbers does the layer carry from
   token to token?

1. [code] **Short initial memories.** Train the Mamba model with step sizes
   initialized between 0.5 and 1 (`dt_min=0.5, dt_max=1.0`), so that every
   state starts with a memory of a few steps.
    1. Does it learn to copy the symbols within the training budget? If so,
       plot the step sizes of the first layer on filler tokens before and
       after training, and explain how the model could have learned long
       memories from gradients that, at the start, vanish over a thousand
       steps. (Hint: where do the symbols closest to the queries sit?)
    1. Repeat with 4,096 fillers, and with the minGRU initialized as in the
       long-range recall experiment of that section.

[Discussions](https://d2l.discourse.group/)

[Discussions](https://d2l.discourse.group/t/36782)